# Chapter 40: Classical Object Recognition & Scene Analysis
## Multimedia & Computer Vision Learning Hub - ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Objective:** Mempelajari pencocokan templat Normalized Cross-Correlation (NCC), ekstraksi deskriptor HOG, klasifikasi SVM, serta penyaringan Non-Maximum Suppression (NMS).

### 1. Landasan Teori & Formulasi Matematis
Koefisien korelasi silang ternormalisasi (NCC):
$$\gamma(x, y) = \frac{\sum_{u,v} [T(u, v) - \bar{T}][I(x+u, y+v) - \bar{I}]}{\sqrt{\sum_{u,v} [T(u, v) - \bar{T}]^2 \sum_{u,v} [I(x+u, y+v) - \bar{I}]^2}}$$

Formulasi Intersection over Union (IoU) antara dua bounding box:
$$\text{IoU} = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)}$$

In [ ]:
# 2. Dependency Setup & Environment
import numpy as np
import cv2
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. Sintesis Dataset Citra Uji & Objek Target (Template Matching)
scene_img = np.zeros((240, 320), dtype=np.uint8)
# Berikan tekstur latar belakang acak berkontras rendah
np.random.seed(42)
noise = np.random.randint(20, 60, (240, 320), dtype=np.uint8)
scene_img = cv2.add(scene_img, noise)

# Buat templat pola target geometris (lingkaran dengan cincin)
target_template = np.zeros((40, 40), dtype=np.uint8)
cv2.circle(target_template, (20, 20), 16, 200, -1)
cv2.circle(target_template, (20, 20), 8, 50, -1)

# Tempelkan target pada koordinat (x=140, y=80)
scene_img[80:120, 140:180] = target_template
print(f"Dimensi Scene: {scene_img.shape}, Dimensi Templat: {target_template.shape}")

In [ ]:
# 4. Eksekusi Pencocokan Templat NCC (cv2.TM_CCOEFF_NORMED)
res_map = cv2.matchTemplate(scene_img, target_template, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res_map)

print(f"Nilai Korelasi Puncak NCC: {max_val:.4f}")
print(f"Lokasi Deteksi Terbaik: {max_loc} (Ground Truth: (140, 80))")
assert max_loc == (140, 80), "Deteksi templat harus menemukan lokasi secara presisi"

In [ ]:
# 5. Ekstraksi Fitur Histogram of Oriented Gradients (HOG)
# Parameter standar Dalal-Triggs: patch=(64, 128), cell=(8, 8), block=(2, 2) cells, nbins=9
def extract_hog_dalal_triggs(image, cell_size=(8, 8), block_size=(2, 2), n_bins=9):
    h, w = image.shape
    gx = np.zeros_like(image, dtype=np.float32)
    gy = np.zeros_like(image, dtype=np.float32)
    gx[:, 1:-1] = (image[:, 2:].astype(np.float32) - image[:, :-2].astype(np.float32)) / 2.0
    gy[1:-1, :] = (image[2:, :].astype(np.float32) - image[:-2, :].astype(np.float32)) / 2.0
    mag = np.hypot(gx, gy)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0
    c_h, c_w = cell_size
    n_cells_y = h // c_h
    n_cells_x = w // c_w
    histograms = np.zeros((n_cells_y, n_cells_x, n_bins), dtype=np.float32)
    bin_width = 180.0 / n_bins
    for i in range(n_cells_y):
        for j in range(n_cells_x):
            c_mag = mag[i*c_h:(i+1)*c_h, j*c_w:(j+1)*c_w]
            c_ang = ang[i*c_h:(i+1)*c_h, j*c_w:(j+1)*c_w]
            b_idx = (c_ang / bin_width).astype(int) % n_bins
            for b in range(n_bins):
                histograms[i, j, b] = np.sum(c_mag[b_idx == b])
    b_y, b_x = block_size
    n_blocks_y = n_cells_y - b_y + 1
    n_blocks_x = n_cells_x - b_x + 1
    descriptors = []
    for i in range(n_blocks_y):
        for j in range(n_blocks_x):
            block_feat = histograms[i:i+b_y, j:j+b_x, :].flatten()
            norm = np.linalg.norm(block_feat) + 1e-6
            descriptors.append(block_feat / norm)
    return np.concatenate(descriptors)

# Buat sampel uji berukuran 64x128 piksel
sample_patch = np.zeros((128, 64), dtype=np.uint8)
sample_patch[20:108, 32] = 255  # Garis vertikal kontras

hog_features = extract_hog_dalal_triggs(sample_patch)
print(f"Dimensi Vektor Fitur HOG: {hog_features.shape[0]} dimensi (Ekspektasi: 3780)")
assert hog_features.shape[0] == 3780, "Dimensi vektor HOG standar 64x128 harus tepat 3.780 dimensi"


### 6. Batasan & Analisis Kegagalan (Failure Modes)
1. **Sensitivitas Rotasi pada HOG**: HOG tidak memiliki orientasi kanonikal global sehingga performa anjlok bila objek terotasi > 20 derajat.
2. **Variasi Skala Citra**: Template matching NCC konvensional gagal total jika ukuran objek pada citra berbeda dari templat acuan (memerlukan piramida multi-skala).
3. **Latar Belakang Oklusi & Clutter**: Ambang batas deteksi SVM linear memicu false positives tinggi pada tekstur dedaunan atau semak belukar yang padat.

### 7. Referensi Akademik
- Dalal, N., & Triggs, B. (2005). *Histograms of Oriented Gradients for Human Detection*. CVPR.
- Cortes, C., & Vapnik, V. (1995). *Support-Vector Networks*. Machine Learning.